<a href="https://colab.research.google.com/github/TejashwaniIndoria/FraudDetection/blob/main/FRAUDDETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gdown gradio scikit-learn pandas

import gdown
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import gradio as gr

print("⬇ Downloading dataset from Google Drive...")
url = "https://drive.google.com/uc?id=1NlcPkLwd8rS899Z0gYMeat43vgp3RJU6"
output = "fraud_data.csv"
gdown.download(url, output, quiet=False)

print("📊 Loading dataset...")
df = pd.read_csv(output)
print(f"✅ Dataset loaded. Shape: {df.shape}")

fraction = 0.05  # Use 5% of data
print(f"⚡ Sampling {fraction*100:.1f}% of dataset...")
df_sampled = df.sample(frac=fraction, random_state=42)
print(f"✅ Sampled shape: {df_sampled.shape}")

print("🔠 Encoding 'type' column...")
df_sampled['type'] = LabelEncoder().fit_transform(df_sampled['type'])
print("✅ Encoding done.")

X = df_sampled[['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']]
y = df_sampled['isFraud']

print("✂ Splitting dataset...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🤖 Training Random Forest model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("✅ Model trained.")

print("🔍 Predicting on test set...")
df_test = X_test.copy()
df_test['Actual Fraud'] = y_test.values
df_test['Predicted Fraud'] = model.predict(X_test)

def fraud_badge(val):
    if val == 0:
        return '<span style="color:white; background-color:#d9534f; padding:4px 8px; border-radius:4px;">⚠ Fraud</span>'
    else:
        return '<span style="color:white; background-color:#5cb85c; padding:4px 8px; border-radius:4px;">✅ Safe</span>'

df_test['Actual Fraud'] = df_test['Actual Fraud'].apply(fraud_badge)
df_test['Predicted Fraud'] = df_test['Predicted Fraud'].apply(fraud_badge)

display_df = df_test[['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'Actual Fraud', 'Predicted Fraud']]
display_df.rename(columns={
    'type': 'Transaction Type',
    'amount': 'Amount',
    'oldbalanceOrg': 'Old Balance Origin',
    'newbalanceOrig': 'New Balance Origin',
    'oldbalanceDest': 'Old Balance Destination',
    'newbalanceDest': 'New Balance Destination'
}, inplace=True)

def show_predictions():
    styles = """
    <style>
    table {width: 100%; border-collapse: collapse; font-family: Arial, sans-serif; background-color: #000000; color: white;}
    th, td {border: 1px solid #222222; padding: 8px; text-align: center;}
    th {background-color: #111111; color: white;}
    tr:hover {background-color: #222222;}
    </style>
    """
    fraud_samples = display_df[display_df['Predicted Fraud'].str.contains("Fraud")].sample(n=50, random_state=42)
    safe_samples = display_df[display_df['Predicted Fraud'].str.contains("Safe")].sample(n=50, random_state=42)
    mixed_samples = pd.concat([fraud_samples, safe_samples]).sample(frac=1, random_state=42)

    return styles + mixed_samples.to_html(escape=False, index=False)

iface = gr.Interface(
    fn=show_predictions,
    inputs=[],
    outputs=gr.HTML(label="Fraud Detection Predictions"),
    title="💳 Fraud Detection Predictions",
    description="Displaying prediction results on sample test transactions (mixed view of fraud and safe)."
)

iface.launch(share=True)

⬇ Downloading dataset from Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1NlcPkLwd8rS899Z0gYMeat43vgp3RJU6
From (redirected): https://drive.google.com/uc?id=1NlcPkLwd8rS899Z0gYMeat43vgp3RJU6&confirm=t&uuid=79c521c4-2f68-4760-8175-8ef3051e6bf1
To: /content/fraud_data.csv
100%|██████████| 494M/494M [00:10<00:00, 47.2MB/s]


📊 Loading dataset...
✅ Dataset loaded. Shape: (6362620, 11)
⚡ Sampling 5.0% of dataset...
✅ Sampled shape: (318131, 11)
🔠 Encoding 'type' column...
✅ Encoding done.
✂ Splitting dataset...
🤖 Training Random Forest model...
✅ Model trained.
🔍 Predicting on test set...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc3d5856806e49edb8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
